In [1]:
!pip install --break-system-packages evaluate trackio 'transformers[torch]' tiktoken sentencepiece 'accelerate>=1.1.0'

Looking in indexes: https://pypi.org/simple, https://packagecloud.io/github/git-lfs/pypi/simple


In [2]:
import os
import json
import zipfile
import requests
import numpy as np

import torch
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
import trackio

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Download X-Stance

In [3]:
dataset_url = "https://github.com/ZurichNLP/xstance/raw/refs/heads/master/data/xstance-data-v1.0.zip"

zipfile_path = "xstance-data-v1.0.zip"
extract_dir = "data"

if not os.path.exists(zipfile_path):
    response = requests.get(dataset_url)
    response.raise_for_status()

    with open(zipfile_path, "wb") as f:
        f.write(response.content)

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zipfile_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

# Load JSONL files

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


train_data = load_jsonl(os.path.join(extract_dir, "train.jsonl"))
valid_data = load_jsonl(os.path.join(extract_dir, "valid.jsonl"))
test_data = load_jsonl(os.path.join(extract_dir, "test.jsonl"))

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(valid_data),
    "test": Dataset.from_list(test_data),
})

# Label mapping

In [5]:
dataset['train'].to_pandas()

,id,language,question_id,question,comment,label,numerical_label,author,topic
0,3808,de,44,"Eine Volksinitiative fordert, dass die Gesamtf...","Eine fixe Grösse verbieten, ist das falsche Mi...",AGAINST,25,20a4c510b64f,Infrastructure & Environment
1,10045,de,739,Sind Sie für eine vollständige Liberalisierung...,Das Arbeitsgesetz regelt die Arbeitszeiten und...,FAVOR,100,f27b54a137b4,Economy
2,10099,de,739,Sind Sie für eine vollständige Liberalisierung...,"Es muss jedoch sicher gestellt werden, dass di...",FAVOR,100,4fab6443143e,Economy
3,10146,de,739,Sind Sie für eine vollständige Liberalisierung...,Die realisierten Liberalisierungen habe ich be...,AGAINST,0,a9f7e55858de,Economy
4,10252,de,739,Sind Sie für eine vollständige Liberalisierung...,"Wenn ein Bedarf dafür besteht, ja. Allerdings ...",FAVOR,75,5ea292041495,Economy
...,...,...,...,...,...,...,...,...,...
45635,168394,fr,12,Approuvez-vous que des dispenses scolaires soi...,"Séparation de l'Etat et de la religion, pas de...",AGAINST,0,80ebaa906d20,Education
45636,168412,fr,12,Approuvez-vous que des dispenses scolaires soi...,"Par principe j'y suis plutôt opposé, cependant...",AGAINST,25,086f43f2bf52,Education
45637,206060,fr,12,Approuvez-vous que des dispenses scolaires soi...,Nous devons respecter coute que coute la laïci...,AGAINST,0,983d48ac9282,Education
45638,206088,fr,12,Approuvez-vous que des dispenses scolaires soi...,Non. Il est important que l'école demeure laïq...,AGAINST,0,4cbe2a34da39,Education


In [6]:
print(dataset["train"].features)
print(dataset["train"][0])

label_names = sorted(set(dataset["train"]["label"]))
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

print("Labels:", label2id)


def encode_labels(example):
    example["labels"] = label2id[example["label"]]
    return example


dataset = dataset.map(encode_labels)

{'id': Value('int64'), 'language': Value('string'), 'question_id': Value('int64'), 'question': Value('string'), 'comment': Value('string'), 'label': Value('string'), 'numerical_label': Value('int64'), 'author': Value('string'), 'topic': Value('string')}
{'id': 3808, 'language': 'de', 'question_id': 44, 'question': 'Eine Volksinitiative fordert, dass die Gesamtfläche der Bauzonen in der Schweiz für die nächsten 20 Jahre auf dem heutigen Stand begrenzt wird. Befürworten Sie dieses Anliegen?', 'comment': 'Eine fixe Grösse verbieten, ist das falsche Mittel', 'label': 'AGAINST', 'numerical_label': 25, 'author': '20a4c510b64f', 'topic': 'Infrastructure & Environment'}
Labels: {'AGAINST': 0, 'FAVOR': 1}


Map: 100%|██████████| 17705/17705 [00:01<00:00, 10267.22 examples/s]


# Tokenizer

In [7]:
model_card = "microsoft/mdeberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_card)


def tokenize_fn(batch):
    return tokenizer(
        batch["question"],
        batch["comment"],
        truncation="only_second",
        max_length=512,
        padding=False,
    )


tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=[
        "question",
        "comment",
        "label",
        "numerical_label",
        "author",
        "topic",
        "language",
        "id",
        "question_id",
    ],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 17705/17705 [00:01<00:00, 11046.14 examples/s]


# Model

In [16]:
num_labels = len(label_names)

model = AutoModelForSequenceClassification.from_pretrained(
    model_card,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.float32,
)
model = model.float()

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 918.30it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
classifier.

# Metrics

In [17]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = f1.compute(
        predictions=preds,
        references=labels,
        average="macro",
    )["f1"]

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
    }

# Training config

In [18]:
BATCH_SIZE = 16
training_args = TrainingArguments(
    output_dir="./mdeberta-v3-xstance",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=850,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    push_to_hub=True,
    max_grad_norm=1.0,
    fp16=False,
    bf16=False,
    hub_model_id="MatteoFasulo/mdeberta-v3-xstance",
    report_to="trackio",
    seed=42,
    data_seed=42,
)

# Trainer

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Train

In [20]:
trainer.train()

* Run finished. Uploading logs to Trackio (please wait...)
* Created new run: merry-cloud-10


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.446591,0.492040,0.788589,0.788522
2,0.339499,0.455061,0.807692,0.807689
3,0.252247,0.508268,0.815334,0.815333


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.57s/it]


* Project data uploaded to bucket: https://huggingface.co/buckets/MatteoFasulo/huggingface-static-26c811-bucket
* Static Space deployed: https://huggingface.co/spaces/MatteoFasulo/huggingface-static-26c811


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.05s/it]


...

Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.64s/it]


* Project data uploaded to bucket: https://huggingface.co/buckets/MatteoFasulo/huggingface-static-26c811-bucket


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


* Static Space deployed: https://huggingface.co/spaces/MatteoFasulo/huggingface-static-26c811
* Run finished. Uploading logs to Trackio (please wait...)


TrainOutput(global_step=8559, training_loss=0.40317999613258604, metrics={'train_runtime': 1522.1963, 'train_samples_per_second': 89.949, 'train_steps_per_second': 5.623, 'total_flos': 1.1053070211774816e+16, 'train_loss': 0.40317999613258604, 'epoch': 3.0})

## Results

In [69]:
trackio.init(project="huggingface")

* Created new run: dainty-sunset-0


In [70]:
valid_results = trainer.evaluate(tokenized_dataset["validation"])
print("Validation results:", valid_results)

test_results = trainer.evaluate(tokenized_dataset["test"])
print("Test results:", test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.252247,0.508268,3,0.815334,0.815333


Validation results: {'eval_loss': 0.5082683563232422, 'eval_accuracy': 0.8153336729495669, 'eval_macro_f1': 0.8153333734281829}


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.252247,0.564309,3,0.791471,0.791191


Test results: {'eval_loss': 0.5643088817596436, 'eval_accuracy': 0.7914713357808528, 'eval_macro_f1': 0.7911907618994465}


In [72]:
trainer.push_to_hub()
model.push_to_hub("MatteoFasulo/mdeberta-v3-xstance")
tokenizer.push_to_hub("MatteoFasulo/mdeberta-v3-xstance")

Processing Files (5 / 5): 100%|██████████|  124kB /  124kB, 11.6kB/s  
New Data Upload: 100%|██████████|  124kB /  124kB, 11.6kB/s  


* Project data uploaded to bucket: https://huggingface.co/buckets/MatteoFasulo/huggingface-static-26c811-bucket


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


* Static Space deployed: https://huggingface.co/spaces/MatteoFasulo/huggingface-static-26c811


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.02s/it]
Processing Files (1 / 1): 100%|██████████| 1.12GB / 1.12GB, 81.1MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.
Processing Files (1 / 1): 100%|██████████| 16.0MB / 16.0MB, 1.49MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/MatteoFasulo/mdeberta-v3-xstance/commit/53f0a4bdf0e3d8b93ec8c1cb3599e9d955bbb2b7', commit_message='Upload tokenizer', commit_description='', oid='53f0a4bdf0e3d8b93ec8c1cb3599e9d955bbb2b7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MatteoFasulo/mdeberta-v3-xstance', endpoint='https://huggingface.co', repo_type='model', repo_id='MatteoFasulo/mdeberta-v3-xstance'), pr_revision=None, pr_num=None)